# Does the adapter actually generate a good patch?

The 31B cannot run on a 6 GB laptop (4-bit base is ~16.5 GB; E2B was the local
workaround), so the adapters get tested where they were trained: Kaggle, 2x T4.

This is an honest A/B, not a demo reel:
  1. load `gemma-4-31b-it-qat-q4a16` in NF4 (the serving checkpoint)
  2. pull `main_lora` / `tool_lora` from the public HF repo we published
  3. take real dev tasks from `tasks.jsonl`
  4. render the prompt exactly as training did (same system prompt, same chat
     template, same `enable_thinking`), then generate with and without the
     adapter and print both side by side

What we are looking for: the adapter's output should be a short `LOCATE/CAUSE/
CHANGE` plan followed by a fenced unified diff. The base model, left to itself,
tends to write prose or invent file paths — that difference is the whole point
of the fine-tune.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["DISABLE_KAGGLE_CACHE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
import sys, time, json, pathlib
try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

In [ ]:
import json as _json, pathlib as _pl, traceback as _tb

WORK = _pl.Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
DIAG_PATH = WORK / "diagnostics.json"
DIAG = []

def diag(step, ok=True, **kw):
    """Append a step result and flush it to disk immediately."""
    rec = {"step": step, "ok": bool(ok)}
    rec.update(kw)
    DIAG.append(rec)
    DIAG_PATH.write_text(_json.dumps(DIAG, indent=1, default=str), encoding="utf-8")
    print(f"[diag] {step} ok={ok} {kw if not ok else ''}", flush=True)
    return rec

def _hook(exc_type, exc, tb):
    diag("UNCAUGHT:" + getattr(exc_type, "__name__", "?"), ok=False,
         error=str(exc)[:800],
         traceback="".join(_tb.format_tb(tb))[-2500:])
    sys.__excepthook__(exc_type, exc, tb)

sys.excepthook = _hook

# NOTE: deliberately NO huggingface_hub import in this cell.
#
# v6 and v7 both installed huggingface_hub==1.33.0 (importlib.metadata confirmed
# it) and then still raised
#   ImportError: cannot import name 'as_extended_path' from
#                'huggingface_hub.utils._paths'
# ...even though 1.32.0+ does provide that symbol. Cause: this cell used to do
# `from huggingface_hub import HfApi` to probe the base repo, which ran BEFORE
# the pip cell and cached the runner's preinstalled (too old) hub in
# sys.modules. The later install replaced the files on disk, but `import
# huggingface_hub` in the same process kept handing back the cached old module.
#
# So: touch os.environ only here, and probe the Hub after the install, from a
# subprocess, so nothing can poison the parent's module cache.
import os as _os
if _os.environ.get("HF_TOKEN"):
    diag("hf_token_present", ok=True)
else:
    diag("hf_token_present", ok=False, note="no HF_TOKEN in the runner environment")


In [ ]:
try:
    import subprocess

    def sh(cmd):
        print("$", cmd, flush=True)
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        print((p.stdout or "")[-2000:])
        if p.returncode != 0:
            print((p.stderr or "")[-2000:])
        return p.returncode

    # Pinned deliberately, and pinned with `==` only.
    #
    # v6 failed here with returncode 2, and the reason was in the output directory:
    # a 0-byte file literally named `=0.23.1,`. `sh()` runs through `shell=True`, so
    # a requirement written as `tokenizers>=0.23.1,<0.24.0` was split by the shell
    # into a stdout redirect to a file named `=0.23.1,` plus an input redirect from
    # `0.24.0`. The range never reached pip, tokenizers stayed at 0.22.2, and
    # transformers 5.17.0 then refused to import because it needs >=0.23.1.
    #
    # Two changes: exact `==` pins (no shell metacharacters at all) and shlex.quote
    # on top, so no future edit can reintroduce this.
    #
    # transformers 5.17.0 also needs `as_extended_path` from huggingface_hub, which
    # no hub below 1.30 provides, while its own metadata claims
    # `huggingface-hub<2.0,>=1.5.0`. --no-deps on transformers is what lets a 1.x
    # hub satisfy it.
    import shlex

    WANT = {
        "transformers": "5.17.0",
        "huggingface_hub": "1.33.0",
        "tokenizers": "0.23.1",
        "safetensors": "0.8.0",
        "accelerate": "1.13.0",
        "peft": "0.21.0",
        "bitsandbytes": "0.50.2",
        "compressed-tensors": "0.19.0",
    }
    specs = [f"{k}=={v}" for k, v in WANT.items() if k != "transformers"]

    rc = sh(f"{shlex.quote(sys.executable)} -m pip uninstall -y torchao")
    rc |= sh(f"{shlex.quote(sys.executable)} -m pip install -q --no-deps "
             f"{shlex.quote('transformers==5.17.0')}")
    rc |= sh(f"{shlex.quote(sys.executable)} -m pip install -q -U "
             + " ".join(shlex.quote(s) for s in specs))
    diag("pip_install", ok=(rc == 0), returncode=rc, specs=specs)

    # Do not trust pip's exit code: report what is ACTUALLY installed and fail here
    # rather than three cells later with an opaque ImportError.
    from importlib import metadata as _md
    actual = {}
    for k in WANT:
        try:
            actual[k] = _md.version(k)
        except Exception:
            actual[k] = None
    bad = {k: (actual[k], v) for k, v in WANT.items() if actual[k] != v}
    diag("stack_versions", ok=(not bad), installed=actual,
         mismatched={k: {"got": g, "want": w} for k, (g, w) in bad.items()})
    if bad:
        raise RuntimeError(f"dependency install did not take: {bad}")

    try:
        import huggingface_hub, transformers, peft, kagglehub
        from transformers import (AutoConfig, AutoProcessor, AutoModelForCausalLM,
                                  BitsAndBytesConfig)
        diag("stack_imports", ok=True, hub=huggingface_hub.__version__,
             transformers=transformers.__version__, peft=peft.__version__,
             kagglehub=getattr(kagglehub, "__version__", "?"))
    except Exception as _e:
        diag("stack_imports", ok=False, error=repr(_e)[:400])
        raise

    # Probe the Hub in a SUBPROCESS. Doing it in-process would work, but a
    # subprocess makes it impossible for this probe to influence - or be influenced
    # by - the module cache the rest of the notebook depends on.
    #
    # `sh()` returns subprocess' returncode (an int), not its stdout, so calling
    # .strip() on its result is the bug that killed v8 with
    #   AttributeError("'int' object has no attribute 'strip'").
    # Capture the stream directly instead.
    import subprocess as _sp, json as _json2
    _probe_src = (
        "import json, huggingface_hub;"
        "from huggingface_hub import HfApi;"
        "a = HfApi();"
        "print('PROBE ' + json.dumps({"
        "'hub': huggingface_hub.__version__,"
        "'repos': {c: (lambda r: {'gated': r.gated, 'private': r.private})(a.model_info(c))"
        "          for c in ['google/gemma-4-31b-it-qat-w4a16-ct', 'google/gemma-4-31b-it']}}))"
    )
    _p = _sp.run([sys.executable, "-c", _probe_src], capture_output=True, text=True, timeout=300)
    _probe_line = next((l for l in (_p.stdout or "").splitlines() if l.startswith("PROBE ")), "")
    if _p.returncode != 0 or not _probe_line:
        diag("hub_probe", ok=False, returncode=_p.returncode,
             stderr=(_p.stderr or "")[-400:])
    else:
        try:
            diag("hub_probe", ok=True, **json.loads(_probe_line[len("PROBE "):]))
        except Exception as _e:
            diag("hub_probe", ok=False, error=repr(_e)[:200], raw=_probe_line[:300])

except BaseException as _e:
    diag('cell_3', ok=False, error=repr(_e)[:400])
    raise
diag('cell_3', ok=True)


In [ ]:
try:
    import torch, transformers, peft, kagglehub
    from transformers import (AutoConfig, AutoProcessor, AutoModelForCausalLM,
                              BitsAndBytesConfig)
    from peft import PeftModel

    print("torch       ", torch.__version__)
    print("transformers", transformers.__version__)
    print("peft        ", peft.__version__)
    print("gpus:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

    WORK = pathlib.Path("/kaggle/working")
    WORK.mkdir(parents=True, exist_ok=True)
except BaseException as _e:
    diag('cell_4', ok=False, error=repr(_e)[:400])
    raise
diag('cell_4', ok=True)


In [ ]:
try:
    # The `google/gemma-4/Other/...` path does not exist on the Hub. The
    # serving checkpoint is the flat id, which is also what the adapter
    # declares in its own base_model_name_or_path.
    BASE_CANDIDATES = [
        "google/gemma-4-31b-it-qat-w4a16-ct",
        "google/gemma-4-31b-it",
    ]
    BASE = None
    for _cand in BASE_CANDIDATES:
        try:
            from huggingface_hub import HfApi
            _i = HfApi().model_info(_cand)
            print("base candidate OK:", _cand, "gated =", _i.gated)
            BASE = _cand
            break
        except Exception as _e:
            print("base candidate FAILED:", _cand, repr(_e)[:120])
    if BASE is None:
        raise RuntimeError(f"no usable base checkpoint among {BASE_CANDIDATES}")
    diag("base_id", ok=True, repo=BASE)
    HF_REPO = "karthik-a/sparkflashxgemma4"

    t0 = time.time()
    try:
        # kagglehub.model_download only understands Kaggle Models
        # handles, so it rejected the Hub repo id outright.
        from huggingface_hub import snapshot_download
        model_path = pathlib.Path(snapshot_download(BASE, max_workers=8))
        print(f"base: {model_path}  {time.time() - t0:.0f}s")
        diag("model_download", ok=True, path=str(model_path))
    except Exception as _e:
        diag("model_download", ok=False, error=repr(_e)[:400])
        raise

    COMP = "gemma-4-developer-agent"
    try:
        cd = pathlib.Path(kagglehub.competition_download(COMP, path="tasks.jsonl"))
        tasks_path = next(cd.rglob("tasks.jsonl"))
    except Exception as e:
        print("kagglehub competition_download failed:", repr(e)[:120])
        DATA = WORK / "data"; DATA.mkdir(parents=True, exist_ok=True)
        sh(f"kaggle competitions download -c {COMP} -f tasks.jsonl -p {DATA}")
        tasks_path = DATA / "tasks.jsonl"
    tasks = [json.loads(l) for l in tasks_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    diag("tasks_loaded", ok=True, n=len(tasks))
    print(f"{len(tasks)} dev tasks")
except BaseException as _e:
    diag('cell_5', ok=False, error=repr(_e)[:400])
    raise
diag('cell_5', ok=True)


## Load the 4-bit base and the published adapter

Pulling from HF rather than re-using local files also proves the repo we pushed
is actually loadable by a third party.

In [ ]:
try:
    from huggingface_hub import hf_hub_download

    t0 = time.time()
    adapter_dir = pathlib.Path(hf_hub_download(HF_REPO, "main_lora/adapter_model.safetensors",
                                               repo_type="model"))
    adapter_cfg = pathlib.Path(hf_hub_download(HF_REPO, "main_lora/adapter_config.json",
                                               repo_type="model"))
    tool_dir = pathlib.Path(hf_hub_download(HF_REPO, "tool_lora/adapter_model.safetensors",
                                            repo_type="model"))
    tool_cfg = pathlib.Path(hf_hub_download(HF_REPO, "tool_lora/adapter_config.json",
                                            repo_type="model"))
    print(f"adapters from HF in {time.time() - t0:.0f}s")
    print(" main:", json.loads(adapter_cfg.read_text())["r"],
          " tool:", json.loads(tool_cfg.read_text())["r"])

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_quant_storage=torch.uint8)

    # The serving checkpoint's config.json declares compressed-tensors quantization.
    # transformers validates that path even when we pass a BitsAndBytesConfig, and
    # that path has no quantized forward (it expands the model to bf16 on first use).
    # Strip it so bitsandbytes NF4 is unambiguously the only quantizer considered.
    serve_cfg = AutoConfig.from_pretrained(model_path)
    print("config quantization_config:", getattr(serve_cfg, "quantization_config", None))
    serve_cfg.quantization_config = None

    t0 = time.time()
    _ngpu = torch.cuda.device_count()
    _tot = [round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2)
              for i in range(_ngpu)]
    diag("gpu_shape", ok=(_ngpu >= 2), device_count=_ngpu,
         total_gib_per_device=_tot,
         names=[torch.cuda.get_device_name(i) for i in range(_ngpu)],
         note=('a 31B in NF4 needs ~16.5-17GB of VRAM; a single 16GB T4 is not enough'
               if _ngpu < 2 else 'two GPUs available'))
    base_model = AutoModelForCausalLM.from_pretrained(
        model_path, config=serve_cfg, quantization_config=bnb,
        dtype=torch.float16, device_map="auto",
        max_memory={**{i: "13GiB" for i in range(torch.cuda.device_count())},
                   "cpu": "26GiB"},
        attn_implementation="sdpa", low_cpu_mem_usage=True)
    print(f"base loaded in {time.time() - t0:.0f}s")
    print("Linear4bit modules:", sum(1 for _, m in base_model.named_modules()
                                    if type(m).__name__ == "Linear4bit"))

    processor = AutoProcessor.from_pretrained(model_path)
    tok = getattr(processor, "tokenizer", processor)
except BaseException as _e:
    diag('cell_7', ok=False, error=repr(_e)[:400])
    raise
diag('cell_7', ok=True)


## Prompts, rendered exactly as in training

In [ ]:
try:
    SYSTEM_MAIN = (
        "You are SparkFlashXGamma4, an autonomous coding agent. You map the code graph and "
        "think step-by-step before acting. You reason briefly, then act. You never guess file "
        "contents: you locate the code first, make the smallest change that fixes the reported "
        "problem, verify it, and finish by calling submit_patch with a unified diff."
    )
    SYSTEM_TOOL = (
        "You are SparkFlashXGamma4 CodeAnalyzer, a read-only explorer. You locate code for "
        "another agent. You never write files, never emit patches, and never propose changes. "
        "You report which files and symbols matter, with line ranges and why."
    )

    def render_user(task):
        repo = str(task.get("repo") or task.get("repo_name") or task.get("instance_id") or "")
        problem = str(task.get("problem_statement") or task.get("issue") or "").strip()
        hints = str(task.get("hints_text") or "").strip()
        parts = ([f"Repository: {repo}"] if repo else []) + ["", problem]
        if hints:
            parts += ["", "Hints:", hints[:1200]]
        return "\n".join(parts).strip()

    def tool_prompt(task):
        return ("Find the code that must change for this task. Do not propose a patch.\n\n"
                + render_user(task))

    def build(system, user):
        return processor.apply_chat_template(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True)
except BaseException as _e:
    diag('cell_9', ok=False, error=repr(_e)[:400])
    raise
diag('cell_9', ok=True)


In [ ]:
try:
    def generate(model, system, user, max_new_tokens=700, temperature=0.2):
        prompt = build(system, user)
        enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens,
                                 temperature=temperature, do_sample=temperature > 0,
                                 pad_token_id=tok.pad_token_id or tok.eos_token_id)
        dt = time.time() - t0
        text = tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=False)
        n_new = out.shape[-1] - enc["input_ids"].shape[-1]
        return text, n_new, dt

    def show(tag, text, n_new, dt, limit=1800):
        print("\n" + "=" * 78)
        print(f"{tag}   |  {n_new} new tokens in {dt:.1f}s  ({n_new / max(dt, 1e-6):.2f} tok/s)")
        print("=" * 78)
        print(text[:limit])
        print("-" * 78)
except BaseException as _e:
    diag('cell_10', ok=False, error=repr(_e)[:400])
    raise
diag('cell_10', ok=True)


## Mid-difficulty task selection

The easiest tasks (1-3 line patches) flatter any adapter: almost any output
format scores well. Take mid-difficulty instead - patches in the 6..60 changed
line band - and spread them across repositories so we are not testing one
codebase's conventions.

In [ ]:
try:
    def changed_lines(p):
        return sum(1 for l in str(p or "").splitlines()
                   if l[:1] in "+-" and not l.startswith(("+++", "---")))

    def files_in(p):
        return [l[6:] for l in str(p or "").splitlines() if l.startswith("--- a/")]

    scored = [(changed_lines(t.get("patch")), t) for t in tasks]
    mid = [(n, t) for n, t in scored if 6 <= n <= 60]
    mid.sort(key=lambda x: x[0])
    print(f"dev tasks by gold-patch size: min={scored[0][0]} "
          f"median={scored[len(scored) // 2][0]} max={scored[-1][0]}")
    print(f"mid-difficulty band 6..60 changed lines: {len(mid)} tasks")

    # one task per repo where possible, prefer the middle of the band
    picked, seen = [], set()
    for n, t in sorted(mid, key=lambda x: abs(x[0] - 20)):
        repo = str(t.get("repo", ""))
        if repo in seen:
            continue
        seen.add(repo)
        picked.append((n, t))
        if len(picked) == 4:
            break

    DEMO = [t for _, t in picked]
    for i, t in enumerate(DEMO):
        print(f"\n[task {i}] {t.get('repo')}  instance={t.get('instance_id')}")
        print(f"  gold patch : {changed_lines(t.get('patch'))} changed lines")
        print(f"  gold files : {files_in(t.get('patch'))[:4]}")
except BaseException as _e:
    diag('cell_12', ok=False, error=repr(_e)[:400])
    raise
diag('cell_12', ok=True)


## Chat harness

`chat()` runs a full multi-turn conversation through the model's own template,
the same way the scoring harness will. We keep the assistant turns we generated
so later turns see real history.

In [ ]:
try:
    def chat(model, system, turns, max_new_tokens=600, temperature=0.2, label=""):
        """turns: list of user strings. Returns (list of assistant replies, stats)."""
        history = [{"role": "system", "content": system}]
        replies, total_new, total_t = [], 0, 0.0
        for i, user in enumerate(turns):
            history.append({"role": "user", "content": user})
            prompt = processor.apply_chat_template(
                history, tokenize=False, add_generation_prompt=True, enable_thinking=True)
            enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
            t0 = time.time()
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=max_new_tokens,
                                     temperature=temperature, do_sample=temperature > 0,
                                     pad_token_id=tok.pad_token_id or tok.eos_token_id)
            dt = time.time() - t0
            n_new = out.shape[-1] - enc["input_ids"].shape[-1]
            reply = tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=False)
            replies.append(reply)
            total_new += n_new
            total_t += dt
            history.append({"role": "assistant", "content": reply})
            print(f"\n{'-' * 78}")
            print(f"{label} turn {i + 1}: {n_new} tokens in {dt:.1f}s "
                  f"({n_new / max(dt, 1e-6):.2f} tok/s)")
            print(f"USER: {user[:300]}")
            print(f"ASSISTANT:\n{reply[:2000]}")
        return replies, {"new_tokens": total_new, "seconds": round(total_t, 1),
                         "tok_s": round(total_new / max(total_t, 1e-6), 2)}

    FOLLOWUPS = {
        "main": "Before I apply that: which exact lines change, and how do I verify "
                "the fix without breaking the existing tests?",
        "tool": "Which of those files should I open first, and why that one?",
    }

    def gold_summary(t):
        print("\nGOLD (for comparison, not shown to the model):")
        print(f"  files  : {files_in(t.get('patch'))}")
        print(f"  hunks  : {len(files_in(t.get('patch')))} file(s), "
              f"{changed_lines(t.get('patch'))} changed lines")
        print(f"  patch  :\n{str(t.get('patch'))[:900]}")
except BaseException as _e:
    diag('cell_14', ok=False, error=repr(_e)[:400])
    raise
diag('cell_14', ok=True)


## Turn 1 — base model vs base + main_lora

Identical prompt, temperature 0.2, same seed. The only difference is the
adapter. We are looking for: does the adapter emit a parseable diff in the
right shape, and does it point LOCATE at the file the gold patch actually
touches?

In [ ]:
try:
    TASKS = DEMO[:3]
    base_stats, adapter_stats, transcript = [], [], {}

    for i, t in enumerate(TASKS):
        print("\n\n" + "#" * 78)
        print(f"# TASK {i}: {t.get('instance_id')}")
        print("#" * 78)
        print(render_user(t)[:1000])
        gold_summary(t)

        replies, st = chat(base_model, SYSTEM_MAIN, [render_user(t)], max_new_tokens=600,
                           label=f"[{i} BASE ]")
        base_stats.append(st)
        transcript[f"{i}_base"] = replies

    # attach the adapter
    t0 = time.time()
    # `adapter_dir.parent` is already the .../main_lora directory, so no
    # "subfolder" kwarg here: peft would look for .../main_lora/main_lora and raise.
    model = PeftModel.from_pretrained(base_model, str(adapter_dir.parent),
                                      is_trainable=False)
    print(f"\nPeftModel.from_pretrained(main_lora) in {time.time() - t0:.0f}s")
    print("lora tensors bound:", sum(1 for n, _ in model.named_parameters() if "lora_" in n))

    for i, t in enumerate(TASKS):
        print("\n\n" + "#" * 78)
        print(f"# TASK {i}  (same prompt, adapter attached)")
        print("#" * 78)
        gold_summary(t)
        replies, st = chat(model, SYSTEM_MAIN, [render_user(t)], max_new_tokens=600,
                           label=f"[{i} ADAPT]")
        adapter_stats.append(st)
        transcript[f"{i}_adapter"] = replies
except BaseException as _e:
    diag('cell_16', ok=False, error=repr(_e)[:400])
    raise
diag('cell_16', ok=True)


## Turn 2 — multi-turn follow-up, adapter only

A coding agent gets talked to. If the adapter only helps on turn 1 and then
collapses, that is a real finding.

In [ ]:
try:
    follow = []
    for i, t in enumerate(TASKS[:2]):
        print("\n\n" + "#" * 78)
        print(f"# TASK {i} TURN 2 (multi-turn, adapter only)")
        print("#" * 78)
        history = [{"role": "system", "content": SYSTEM_MAIN},
                   {"role": "user", "content": render_user(t)},
                   {"role": "assistant", "content": transcript[f"{i}_adapter"][0]}]
        prompt = processor.apply_chat_template(
            history + [{"role": "user", "content": FOLLOWUPS["main"]}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True)
        enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=400, temperature=0.2, do_sample=True,
                                 pad_token_id=tok.pad_token_id or tok.eos_token_id)
        reply = tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=False)
        follow.append(reply)
        print(f"USER: {FOLLOWUPS['main']}")
        print(f"ASSISTANT:\n{reply[:1800]}")

    transcript["followups"] = follow
except BaseException as _e:
    diag('cell_18', ok=False, error=repr(_e)[:400])
    raise
diag('cell_18', ok=True)


## Turn 3 — tool_lora: localization, never a patch

Unload main_lora, attach tool_lora, and check it reports files/symbols instead
of trying to edit anything. Scored against the gold files.

In [ ]:
try:
    model = model.unload()
    base_model = model
    # `adapter_dir.parent` is already the .../tool_lora directory, so no
    # "subfolder" kwarg here: peft would look for .../tool_lora/tool_lora and raise.
    model = PeftModel.from_pretrained(base_model, str(tool_dir.parent),
                                      is_trainable=False)
    print("tool_lora attached")

    tool_report = {}
    for i, t in enumerate(TASKS[:2]):
        print("\n\n" + "#" * 78)
        print(f"# TASK {i} — tool_lora localization")
        print("#" * 78)
        replies, st = chat(model, SYSTEM_TOOL, [tool_prompt(t)], max_new_tokens=400,
                           label=f"[{i} TOOL ]")
        text = replies[0]
        gold = files_in(t.get("patch"))
        hits = [f for f in gold if f in text]
        tool_report[f"{i}_tool"] = {"reply": text, "gold_files": gold, "hits": hits}
        print(f"\nGOLD files : {gold}")
        print(f"HITS        : {hits}  ({len(hits)}/{len(gold)})")
except BaseException as _e:
    diag('cell_20', ok=False, error=repr(_e)[:400])
    raise
diag('cell_20', ok=True)


## Summary

In [ ]:
try:
    def fmt_diff(text):
        return "```diff" in text

    summary = {
        "base": {"gen_stats": base_stats,
                 "emitted_diff_block": [fmt_diff(v[0]) for v in
                                        (transcript[f"{i}_base"] for i in range(len(TASKS)))]},
        "adapter": {"gen_stats": adapter_stats,
                    "emitted_diff_block": [fmt_diff(transcript[f"{i}_adapter"][0])
                                           for i in range(len(TASKS))]},
        "tool_lora_file_hits": {k: f"{len(v['hits'])}/{len(v['gold_files'])}"
                                for k, v in tool_report.items()},
        "tasks": [t.get("instance_id") for t in TASKS],
    }
    print("\n" + json.dumps(summary, indent=1))

    (WORK / "chat_transcript.json").write_text(
        json.dumps({"summary": summary, "transcript": transcript,
                    "tool": tool_report}, indent=1), encoding="utf-8")
    print("\nwrote", WORK / "chat_transcript.json")
except BaseException as _e:
    diag('cell_22', ok=False, error=repr(_e)[:400])
    raise
diag('cell_22', ok=True)
